In [9]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
valid_bookings = pd.read_parquet(BASE_DIR / r"_2_feature_engineering+momentum\start\valid_bookings_with_currency_and_google_restaurants_without_duplicates.parquet")
marketing_activities = pd.read_csv("master_marketing_activties.csv")


In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------
# STEP 0A — Normalize booking datetime
# -----------------------
bookings = valid_bookings.copy()

# Prefer created_at if it's the booking creation timestamp; otherwise use start_time.
# (You can switch to 'start_time' if that's the actual dining time and you prefer that.)
bookings["booking_datetime"] = pd.to_datetime(bookings["created_at"], errors="coerce")

# If created_at missing, fallback to start_time
fallback = pd.to_datetime(bookings["start_time"], errors="coerce")
bookings["booking_datetime"] = bookings["booking_datetime"].fillna(fallback)

bookings = bookings.dropna(subset=["restaurant_id", "booking_datetime"]).copy()
bookings["restaurant_id"] = pd.to_numeric(bookings["restaurant_id"], errors="coerce")

# Optional: keep only active (if that's what you want)
# bookings = bookings.loc[bookings["active"].astype(bool)].copy()

# -----------------------
# STEP 0B — Normalize marketing activity times + restaurant_id
# -----------------------
mkt = marketing_activities.copy()

mkt["activity_start"] = pd.to_datetime(mkt["activity_start"], errors="coerce")
mkt["activity_end"] = pd.to_datetime(mkt["activity_end"], errors="coerce")

# Create ONE restaurant_id column (covers CRM/KOL + FB multi-ids)
mkt["restaurant_id"] = (
    pd.to_numeric(mkt.get("crm_restaurant_id"), errors="coerce")
    .combine_first(pd.to_numeric(mkt.get("kol_restaurant_id"), errors="coerce"))
    .combine_first(pd.to_numeric(mkt.get("fb_restaurant_id"), errors="coerce"))
    .combine_first(pd.to_numeric(mkt.get("fb_restaurant_id_2"), errors="coerce"))
    .combine_first(pd.to_numeric(mkt.get("fb_restaurant_id_3"), errors="coerce"))
    .combine_first(pd.to_numeric(mkt.get("fb_restaurant_id_4"), errors="coerce"))
    .combine_first(pd.to_numeric(mkt.get("fb_restaurant_id_5"), errors="coerce"))
)

# Drop any rows without proper window or restaurant
mkt = mkt.dropna(subset=["activity_id", "restaurant_id", "activity_start", "activity_end"]).copy()

# Ensure start <= end
mkt = mkt.loc[mkt["activity_end"] >= mkt["activity_start"]].copy()

# Keep only columns needed for attribution (add more if you want)
mkt_exposure = mkt[[
    "activity_id", "channel", "restaurant_id",
    "activity_start", "activity_end",
    "crm_campaign_name", "crm_topic", "crm_audience",
    "kol_platform", "kol_username", "kol_post_url",
    "fb_ad_name"
]].copy()

# -----------------------
# STEP 0C — Interval join: bookings ↔ marketing exposure windows
# -----------------------
# 1) merge on restaurant_id to create candidates
cand = bookings[["id", "restaurant_id", "booking_datetime", "revenue_thb", "total_guests"]].merge(
    mkt_exposure,
    on="restaurant_id",
    how="left",
    suffixes=("_booking", "_mkt")
)

# 2) keep only where booking is inside [start, end]
attrib = cand[
    (cand["booking_datetime"] >= cand["activity_start"]) &
    (cand["booking_datetime"] <= cand["activity_end"])
].copy()

# -----------------------
# STEP 0D — Resolve overlaps (one booking → one activity)
# Choose the closest activity_start (most plausible driver)
# -----------------------
attrib["time_from_start_hours"] = (attrib["booking_datetime"] - attrib["activity_start"]).dt.total_seconds() / 3600.0
attrib["abs_time_from_start_hours"] = attrib["time_from_start_hours"].abs()

attrib = attrib.sort_values(["id", "abs_time_from_start_hours"])
attrib_1 = attrib.drop_duplicates(subset=["id"], keep="first").copy()

# -----------------------
# OUTPUTS
# -----------------------
OUT_DIR = Path.cwd()  # change if you want

# 1) booking-level attribution
attrib_1.to_csv(OUT_DIR / "booking_attribution_one_touch.csv", index=False)

# 2) optional: many-touch (if you want to analyse overlap)
attrib.to_csv(OUT_DIR / "booking_attribution_all_touches.csv", index=False)

# 3) quick sanity checks
print("Bookings:", bookings.shape)
print("Marketing exposures:", mkt_exposure.shape)
print("Attributed bookings (one-touch):", attrib_1.shape)

print("\nShare of bookings attributed:")
print(attrib_1["channel"].value_counts(dropna=False, normalize=True).head(10))

print("\nTop campaigns by attributed bookings:")
print(attrib_1.groupby(["channel", "activity_id"]).size().sort_values(ascending=False).head(10))


Bookings: (237458, 46)
Marketing exposures: (2932, 12)
Attributed bookings (one-touch): (67998, 18)

Share of bookings attributed:
channel
FB     0.915674
KOL    0.046207
CRM    0.038119
Name: proportion, dtype: float64

Top campaigns by attributed bookings:
channel  activity_id    
FB       FB_74ea7e67c2cf    1871
         FB_11667875402b    1787
         FB_ce460910e554    1606
         FB_5be575a02ca8    1598
         FB_861e98504af0    1576
         FB_8e16e01f00ae    1402
         FB_789cf0613e3a    1397
         FB_2f8582cbce04    1310
         FB_1963741663dc    1298
         FB_c808e3ba40ae    1227
dtype: int64
